# Communication API

First look at the ADEME API: we get the DPE data (existing and new buildings) and compare the two datasets.

## 1. Request

Useful functions for the rest of the notebook.

In [7]:
import pandas as pd
import numpy as np
import requests
import json

In [8]:
url_api_dpe_existants = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines"
url_api_dpe_neufs = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe02neuf/lines"

In [9]:
def req_api(url:str,page:int=None,select:str=None,q:str=None,q_field:str=None,qs:str=None,params:dict=None)-> pd.DataFrame:
    """ function req api and return a dataframe with the data
    @param url: str, url of the api
    @param page: int, page number to request
    @param select: str, fields to select
    @param q: str, query string
    @param q_field: str, field to query
    @param qs: str, query string
    @param params: dict, additional parameters
    @return: pd.DataFrame, dataframe with the data
    """
    if params is None:
        params={
            "page": page if page is not None else None,
            "size": size if size is not None else None,
            "select": select if select is not None else None,
            "q": q if q is not None else None,
        "q_fields": q_fields if q_fields is not None else None,
        "qs": qs if qs is not None else None
    }
        
    rep = requests.get(url, params=params)
    try:
        rep.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"Erreur lors de la requête : {rep.status_code} - {e}")
        return pd.DataFrame()
    if rep.status_code == 200:
        content = rep.json()
        df = pd.DataFrame(content['results'])
        print(f"nb line : {content['total']}")
        print(f"size data collected : {len(df)}, {len(df.columns) if not df.empty else 0}")
        return df

    

## 2. API test

### 2.1 Unique columns of each dataset


In [24]:
params = {
    "page": 1,
    "size": 1000,
}
df_dpe_existants = req_api(url_api_dpe_existants, params=params)
print(f"Nb unique columns for 'dpe existant' : {df_dpe_existants.columns.unique().value_counts().sum()}")

nb line : 15653655
size data collected : 1000, 216
Nb unique columns for 'dpe existant' : 216


In [25]:
df_dpe_neufs = req_api(url_api_dpe_neufs, params=params)
print(f"Nb unique columns for 'dpe neuf' : {df_dpe_neufs.columns.unique().value_counts().sum()}")

nb line : 1432651
size data collected : 1000, 184
Nb unique columns for 'dpe neuf' : 184


In [39]:
def compare_columns(df1: pd.DataFrame, df2: pd.DataFrame) -> None:
    """ function to compare columns of two dataframes
    @param df1: pd.DataFrame, first dataframe
    @param df2: pd.DataFrame, second dataframe
    """
    unique_columns = df1.columns.unique().tolist() + df2.columns.unique().tolist()
    print("nb unique columns in both datasets : ", len(set(unique_columns)))

    nb_columns_df1_not_in_df2 = len(set(df1.columns.unique().tolist()) - set(df2.columns.unique().tolist()))
    print("nb columns in df1 not in df2 : ", nb_columns_df1_not_in_df2)
   
    print("*" * 10)

    nb_columns_df2_not_in_df1 = len(set(df2.columns.unique().tolist()) - set(df1.columns.unique().tolist()))
    print("nb columns in df2 not in df1 : ", nb_columns_df2_not_in_df1)

def print_compare_columns_details(df1: pd.DataFrame, df2: pd.DataFrame) -> None:
    """ 
    @param df1: pd.DataFrame, first dataframe
    @param df2: pd.DataFrame, second dataframe
    @param file_path: str, path to the output file
    """
    columns_df1_not_in_df2 = set(df1.columns.unique().tolist()) - set(df2.columns.unique().tolist())
    columns_df2_not_in_df1 = set(df2.columns.unique().tolist()) - set(df1.columns.unique().tolist())

    print(f"Columns in df1 not in df2 : {len(columns_df1_not_in_df2)}")
    print(f"Columns name: ")
    for col in columns_df1_not_in_df2:
        print(f"- {col}")

    print("*" * 10)
    
    print(f"Columns in df2 not in df1 : {len(columns_df2_not_in_df1)}")
    print(f"Columns name: ")
    for col in columns_df2_not_in_df1:
        print(f"- {col}")


In [41]:
compare_columns(df_dpe_existants, df_dpe_neufs)


nb unique columns in both datasets :  220
nb columns in df1 not in df2 :  36
**********
nb columns in df2 not in df1 :  4


In [40]:
print_compare_columns_details(df_dpe_existants, df_dpe_neufs)

Columns in df1 not in df2 : 36
Columns name: 
- electricite_pv_autoconsommee
- description_generateur_chauffage_n1_installation_n2
- type_generateur_n2_installation_n1
- surface_totale_capteurs_pv
- type_energie_generateur_n2_installation_n1
- usage_generateur_n2_installation_n2
- date_installation_generateur_n1_ecs_n1
- type_ventilation
- systeme_production_electricite_origine_renouvelable
- usage_generateur_n2_installation_n1
- conso_ef_generateur_n1_ecs_n1
- numero_immatriculation_copropriete
- facteur_couverture_solaire_n1
- periode_installation_generateur_froid
- surface_ventilee
- type_energie_climatisation
- production_ecs_solaire_installation_n1
- qualite_isolation_plancher_haut_comble_perdu
- description_generateur_chauffage_n2_installation_n1
- numero_rpls_logement
- conso_chauffage_generateur_n1_installation_n1
- conso_chauffage_generateur_n2_installation_n1
- qualite_isolation_plancher_haut_toit_terrasse
- cop_generateur_n1_ecs_n1
- type_generateur_n2_installation_n2
- nomb

**Result:** the two datasets do not have exactly the same columns.

Only in **DPE existants** (36 columns):
- `electricite_pv_autoconsommee`
- `description_generateur_chauffage_n1_installation_n2`
- `type_generateur_n2_installation_n1`
- `surface_totale_capteurs_pv`
- `type_energie_generateur_n2_installation_n1`
- `usage_generateur_n2_installation_n2`
- `date_installation_generateur_n1_ecs_n1`
- `type_ventilation`
- `systeme_production_electricite_origine_renouvelable`
- `usage_generateur_n2_installation_n1`
- `conso_ef_generateur_n1_ecs_n1`
- `numero_immatriculation_copropriete`
- `facteur_couverture_solaire_n1`
- `periode_installation_generateur_froid`
- `surface_ventilee`
- `type_energie_climatisation`
- `production_ecs_solaire_installation_n1`
- `qualite_isolation_plancher_haut_comble_perdu`
- `description_generateur_chauffage_n2_installation_n1`
- `numero_rpls_logement`
- `conso_chauffage_generateur_n1_installation_n1`
- `conso_chauffage_generateur_n2_installation_n1`
- `qualite_isolation_plancher_haut_toit_terrasse`
- `cop_generateur_n1_ecs_n1`
- `type_generateur_n2_installation_n2`
- `nombre_module`
- `description_generateur_chauffage_n2_installation_n2`
- `presence_production_pv`
- `conso_chauffage_generateur_n2_installation_n2`
- `type_energie_generateur_n2_installation_n2`
- `conso_chauffage_generateur_n1_installation_n2`
- `apport_solaire_saison_chauffe`
- `apport_interne_saison_chauffe`
- `apport_interne_saison_froide`
- `description_generateur_chauffage_n1_installation_n1`
- `apport_solaire_saison_froide`

Only in **DPE neufs** (4 columns):
- `deperditions_totales_batiment`
- `besoin_ecs_logement`
- `besoin_ecs_batiment`
- `deperditions_totales_logement`

### 2.2 Concat data

We keep only the columns common to both datasets.

In [42]:
# Find the common columns between the two datasets
common_columns = set(df_dpe_existants.columns.unique().tolist()).intersection(set(df_dpe_neufs.columns.unique().tolist()))
print(f"Common columns between the two datasets: {len(common_columns)}")
print(f"Columns name: ")
for col in common_columns:
    print(f"- {col}")

Common columns between the two datasets: 180
Columns name: 
- classe_inertie_batiment
- adresse_complete_brut
- conso_5_usages_ef_energie_n1
- deperditions_baies_vitrees
- version_dpe
- emission_ges_5_usages_par_m2
- complement_adresse_batiment
- configuration_installation_ecs_n1
- coordonnee_cartographique_x_ban
- conso_5_usages_par_m2_ep
- type_emetteur_installation_chauffage_n1
- type_energie_principale_chauffage
- conso_ef_installation_ecs_n1
- cout_chauffage_energie_n3
- emission_ges_ecs_energie_n1
- conso_refroidissement_ep
- appartement_non_visite
- deperditions_renouvellement_air
- indicateur_confort_ete
- cout_eclairage
- cout_ecs
- description_generateur_n1_ecs_n1
- deperditions_murs
- etiquette_ges
- numero_dpe_immeuble_associe
- configuration_installation_chauffage_n2
- _score
- emission_ges_eclairage
- cout_auxiliaires
- surface_habitable_immeuble
- volume_stockage_generateur_n1_ecs_n1
- emission_ges_5_usages_energie_n1
- code_insee_ban
- surface_habitable_logement
- emiss

In [46]:
df_dpe_existants['_id'].head(2)

0    8f6d7ec59cb91e4a79c294d5789c74bad817fd9d79ed29...
1    f27777e73bf74a3f5c31195229ee94ca6ddd2e4bc7c043...
Name: _id, dtype: str

In [47]:
df_dpe_existants['numero_dpe'].head(2)

0    2611E0031228S
1    2659E0039476U
Name: numero_dpe, dtype: str

Useful columns for the concat:
- `numero_dpe`
- `_id`

### 2.3 Data type

In [57]:
df_dpe_existants.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 216 entries, configuration_installation_chauffage_n2 to conso_chauffage_generateur_n2_installation_n2
dtypes: float64(113), int64(4), object(1), str(98)
memory usage: 1.6+ MB


In [58]:
df_dpe_neufs.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 184 entries, numero_voie_ban to surface_tertiaire_immeuble
dtypes: float64(99), int64(5), object(1), str(79)
memory usage: 1.4+ MB


In [ ]:
def get_columns_details(df: pd.DataFrame) -> pd.DataFrame:
    """ function to get columns details of a dataframe
    @param df: pd.DataFrame, dataframe to get columns details
    @return: pd.DataFrame, dataframe with columns details
    """
    columns_details = []
    for col in df.columns.unique().tolist():
        col_details = {
            "column_name": col,
            "data_type": df[col].dtype,
            "nb_unique_values": df[col].nunique(),
            "nb_missing_values": df[col].isnull().sum(),
            "missing_values_percentage": df[col].isnull().sum() / len(df) * 100,
        }
        columns_details.append(col_details)
    return pd.DataFrame(columns_details)

In [ ]:
df_details_neufs= get_columns_details(df_dpe_existants).sort_values(by="nb_missing_values", ascending=False)
df_details_existant= get_columns_details(df_dpe_existants).sort_values(by="nb_missing_values", ascending=False)

In [64]:
df_details_existant.head()

,column_name,data_type,nb_unique_values,nb_missing_values,missing_values_percentage
143,_score,object,0,1000,100.0
209,numero_rpls_logement,str,1,999,99.9
211,description_generateur_chauffage_n2_installati...,str,2,997,99.7
215,conso_chauffage_generateur_n2_installation_n2,float64,3,997,99.7
213,type_generateur_n2_installation_n2,str,2,997,99.7


In [65]:
df_details_neufs.head()

,column_name,data_type,nb_unique_values,nb_missing_values,missing_values_percentage
143,_score,object,0,1000,100.0
209,numero_rpls_logement,str,1,999,99.9
211,description_generateur_chauffage_n2_installati...,str,2,997,99.7
215,conso_chauffage_generateur_n2_installation_n2,float64,3,997,99.7
213,type_generateur_n2_installation_n2,str,2,997,99.7


Needs to be considered:
- `_score` : type object and no value


In [68]:
print("Columns with at least 500 missing values:")
for name, df in [("existant", df_details_existant), ("neufs", df_details_neufs)]:
    col_sus = ', '.join(df.loc[df['nb_missing_values'] >= 500, 'column_name'])
    print(f"{name}: {col_sus}")
    print(f"nb : {len(df.loc[df['nb_missing_values'] >= 500])}")


Columns with at least 500 missing values:
existant: _score, numero_rpls_logement, description_generateur_chauffage_n2_installation_n2, conso_chauffage_generateur_n2_installation_n2, type_generateur_n2_installation_n2, usage_generateur_n2_installation_n2, type_energie_generateur_n2_installation_n2, surface_tertiaire_immeuble, numero_immatriculation_copropriete, electricite_pv_autoconsommee, production_ecs_solaire_installation_n1, facteur_couverture_solaire_n1, cout_total_5_usages_energie_n3, conso_chauffage_ef_energie_n3, conso_ecs_ef_energie_n3, cout_ecs_energie_n3, emission_ges_5_usages_energie_n3, emission_ges_ecs_energie_n3, conso_5_usages_ef_energie_n3, cout_chauffage_energie_n3, emission_ges_chauffage_energie_n3, type_energie_n3, nombre_module, surface_totale_capteurs_pv, typologie_logement, position_logement_dans_immeuble, cop_generateur_n1_ecs_n1, type_generateur_n2_installation_n1, type_energie_generateur_n2_installation_n1, conso_chauffage_generateur_n2_installation_n1, usage_

At least 65 columns with at least 50% missing values. (Need to be tested with more row though)

## 3. Conclusion

- **Data access:** the ADEME API works for both datasets (`dpe03existant`: ~15.6M rows, `dpe02neuf`: ~1.4M rows). The analysis was run on a sample of 1000 rows per dataset.
- **Structure:** 216 columns for existing buildings, 184 for new buildings. 180 columns are common to both. 36 columns exist only in *existants* (secondary generators, PV, ventilation, solar gains...) and 4 only in *neufs*.
- **Merging:** the two datasets can be concatenated on the common columns, with `numero_dpe` / `_id` as identifiers.
- **Types:** mostly numeric (`float64`) and text (`str`) columns. `_score` is empty and can be dropped.
- **Missing values:** 65 columns have at least 50% missing values in the sample. Need to confirm the missing-value rates on a larger sample.


